# Research notebook



In [ ]:
# Portable project paths; this cell does not load a model.
from pathlib import Path
import sys

_candidates = (Path.cwd(), *Path.cwd().parents)
_project_root = next((p for p in _candidates if (p / "project_paths.py").is_file()), None)
if _project_root is None:
    raise RuntimeError("Start Jupyter from the retrieval repository or one of its subdirectories.")
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from project_paths import PICTURE_DIR, GALLERY_DIR, OUTPUT_DIR, LLAVA_MODEL, TARGET_CLASSES

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [1]:
import os
from PIL import Image
import torch
from transformers import BertTokenizer, BertForSequenceClassification, CLIPProcessor, CLIPModel
import numpy as np
from pathlib import Path

# 模型初始化（固定设备并启用FP16）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
text_tokenizer = BertTokenizer.from_pretrained("IDEA-CCNL/Taiyi-CLIP-Roberta-large-326M-Chinese")
text_encoder = BertForSequenceClassification.from_pretrained(
    "IDEA-CCNL/Taiyi-CLIP-Roberta-large-326M-Chinese"
).to(device).half().eval()
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).half().eval()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")  # 降低分辨率

In [2]:
def subfolder(exclude):
    root = PICTURE_DIR
    return [str(item) for subdir in root.iterdir()
            if subdir.is_dir()
            for item in subdir.iterdir()
            if item.is_dir() and item!=(PICTURE_DIR / exclude / exclude)]

def load_images(folder_path):
    """原函数：一次性加载所有图片（保留原逻辑）"""
    images = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            with Image.open(file_path) as img:
                images.append(img.copy())
        except Exception as e:
            print(f"无法打开文件 {filename}: {e}")
    return images

def create_path(figure):
    positive_path=str(PICTURE_DIR / figure / figure)
    negative_path=subfolder(figure)
    return positive_path,negative_path

def create_data(p,n):
    """原函数：一次性加载所有图片（保留原逻辑）"""
    P=load_images(p)
    N=[]
    for a in n:
        b=load_images(a)
        N=N+b
    return P,N

def prediction(data, y, batch_size=128):  # 添加batch_size参数，默认32
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clip_model.to(device)
    text_encoder.to(device)

    translate = TARGET_CLASSES
    query = translate[y]
    query_texts = [f'这是一张{translate[y]}的图片', f'这是一张其他的图片']

    text = text_tokenizer(query_texts, return_tensors='pt', padding=True)['input_ids'].to(device)

    text_features = text_encoder(text).logits
    text_features = text_features / text_features.norm(dim=1, keepdim=True)

    all_probs = []
    num_images = len(data)

    # 分批处理图像
    for i in range(0, num_images, batch_size):
        batch_data = data[i:i+batch_size]

        with torch.no_grad():
            image = processor(images=batch_data, return_tensors="pt").to(device)
            image_features = clip_model.get_image_features(**image)
            image_features = image_features / image_features.norm(dim=1, keepdim=True)

            logit_scale = clip_model.logit_scale.exp()
            logits_per_image = logit_scale * image_features @ text_features.t()
            probs = logits_per_image.softmax(dim=-1).cpu().numpy()[:, 0]
            all_probs.extend(probs)

    return np.array(all_probs)

def score(figure):
    positive_path,negative_path=create_path(figure)
    positive_data,negative_data=create_data(positive_path,negative_path)
    Pscore = prediction(positive_data, figure, batch_size=200)
    Nscore = prediction(negative_data, figure, batch_size=200)
    return Pscore,Nscore
def evaluation(threshold,Pscore,Nscore):
    Pclass=[1 if x>=threshold else 0 for x in Pscore]
    Nclass=[1 if x>=threshold else 0 for x in Nscore]
    TP=Pclass.count(1)
    FN=Pclass.count(0)
    FP=Nclass.count(1)
    if TP+FP!=0:
        Precision=TP/(TP+FP)
    if TP+FP==0:
        Precision=0
    if TP+FN!=0:
        Recall=TP/(TP+FN)
    if TP+FN==0:
        Recall=0
    return Precision, Recall



In [3]:
picture = list(TARGET_CLASSES)
Pscore={k:[] for k in picture}
Nscore={k:[] for k in picture}
for figure in picture:
    print(figure)
    pscore,nscore=score(figure)
    Pscore[figure]=Pscore[figure]+pscore.tolist()
    Nscore[figure]=Nscore[figure]+nscore.tolist()

Dog


Piano
Erhu
Porcelain
Duck


0.7107438016528925 0.86
0.6781609195402298 0.59
0.6210045662100456 0.68
0.5821917808219178 0.85
0.7887931034482759 0.915
{'Dog': np.float64(0.88), 'Piano': np.float64(0.89), 'Erhu': np.float64(0.91), 'Porcelain': np.float64(0.96), 'Duck': np.float64(0.9)} {'Dog': 0.7782805429864253, 'Piano': 0.6310160427807486, 'Erhu': 0.6491646778042959, 'Porcelain': 0.6910569105691057, 'Duck': 0.8472222222222223}

0.25722543352601157 0.445
0.30617283950617286 0.62
0.432258064516129 0.67
0.3215339233038348 0.545
0.222007722007722 0.575
{'Dog': np.float64(0.92), 'Piano': np.float64(0.98), 'Erhu': np.float64(0.9500000000000001), 'Porcelain': np.float64(0.92), 'Duck': np.float64(0.8300000000000001)} {'Dog': 0.32600732600732596, 'Piano': 0.4099173553719009, 'Erhu': 0.5254901960784314, 'Porcelain': 0.4044526901669759, 'Duck': 0.3203342618384401}

In [5]:
# 每个类别卡阈值
Threshold0 = {k: 0 for k in picture}
f1_score = {k: 0 for k in picture}

for figure, x in zip(picture, range(len(picture))):
    pre = 0
    rec = 0
    for threshold in np.arange(0.0, 1.01, 0.01):
        precision, recall = evaluation(threshold, Pscore[figure], Nscore[figure])
        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * (precision * recall) / (precision + recall)
        if f1 >= f1_score[figure]:
            Threshold0[figure] = threshold
            f1_score[figure] = f1
            pre = precision
            rec = recall
    print(f'{figure},precision:{pre:.4f},recall:{rec:.4f},f1:{f1_score[figure]:.4f}')

Dog,precision:0.9848,recall:0.9700,f1:0.9773
Piano,precision:0.6575,recall:0.9600,f1:0.7805
Erhu,precision:0.9505,recall:0.8650,f1:0.9058
Porcelain,precision:0.7462,recall:0.9850,f1:0.8491
Duck,precision:0.7941,recall:0.9450,f1:0.8630


In [6]:
# 卡整体阈值

max_macro_f1 = 0
Threshold = 0
for threshold in np.arange(0.0, 1.01, 0.01):
    f1_score = {k: 0 for k in picture}
    for figure, x in zip(picture, range(len(picture))):
        precision, recall = evaluation(threshold,Pscore[figure], Nscore[figure])
        if precision + recall == 0:
            f1_score[figure] = 0
        else:
            f1_score[figure] = 2 * (precision * recall) / (precision + recall)
    macro_f1 = sum([value for value in f1_score.values()]) / len(f1_score)
    if macro_f1 >= max_macro_f1:
        Threshold = threshold
        max_macro_f1 = macro_f1

In [7]:
print(max_macro_f1)

0.8253329255398976


In [8]:
Pclass={k:[] for k in picture}
Nclass={k:[] for k in picture}
for x in picture:
    print(x,Threshold0[x])
    pclass=[1 if y>=Threshold0[x] else 0 for y in Pscore[x]]
    nclass=[1 if y>=Threshold0[x] else 0 for y in Nscore[x]]
    Pclass[x]=pclass
    Nclass[x]=nclass

Dog 0.99
Piano 1.0
Erhu 1.0
Porcelain 0.99
Duck 1.0


In [9]:
print(Pclass)

{'Dog': [1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1], 'Piano': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [12]:
import pickle
import pandas as pd
with open(OUTPUT_DIR / '中文Clip每类正样本预测结果.pkl', 'wb') as f:
    pickle.dump(Pclass, f)
with open(OUTPUT_DIR / '中文Clip每类负样本预测结果.pkl', 'wb') as f:
    pickle.dump(Nclass, f)